# v18 セレクタ動物園: 10手法

PFの96候補/坑井から最終TVTを作る『選択』を、問題設定もモデルも変えて 10通り試す。
共通: 入力=候補(と特徴/系列), 出力=坑井ごとTVT, 評価=overall RMSE, CV=GroupKFold by well。
基準: oracle≈4.62 / lik-ens≈9.35 / v9=6.946。

1. **LGBM-reg**: 各候補のRMSEを回帰 → top-k選択
2. **LGBM-rank**: LambdaMART(坑井内順位)
3. **LGBM-clf**: top-q good/bad 分類 → 確率で選択
4. **MLP-reg**: 表形式MLPでRMSE回帰
5. **MLP-weight**: 特徴→softmax重み→加重平均(end-to-end RMSE)
6. **SetAttn-weight**: 候補間アテンション→重み(コンセンサス考慮)
7. **CNN-cand**: 候補の系列をCNN→RMSE回帰→選択
8. **CNN-permd-gate**: 各mdで候補に重み→per-md加重(学習版多数決)
9. **CNN-traj**: 候補統計を画像化→CNNでTVT軌跡を直接回帰
10. **Stacking**: 1-9のOOFをNNLSブレンド

> 実行はユーザー。神経系はモデル小・epoch少なめ(調整可)。重い場合は N_WELLS で小さく。

In [1]:
import os, glob
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from scipy.stats import rankdata
from scipy.optimize import nnls
from sklearn.model_selection import GroupKFold
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PART_DIR = r'C:\Users\kosaka256\Documents\rogii_claude\v18\artifacts_v18\particles'
N_WELLS = None   # None=全件, 整数で先頭N(デバッグ用)

def load_wells(n=N_WELLS):
    files = sorted(glob.glob(os.path.join(PART_DIR, '*.npz')))
    if n: files = files[:n]
    out = []
    for f in files:
        d = np.load(f); w = {k: d[k] for k in d.files}; w['wid'] = os.path.basename(f)[:-4]
        out.append(w)
    return out
WELLS = load_wells()
print('wells:', len(WELLS), '| cand:', WELLS[0]['cand'].shape)

def pooled_rmse(errs):
    e = np.concatenate(errs); e = e[np.isfinite(e)]; return float(np.sqrt(np.mean(e ** 2)))
def lik_ensemble(w, scale=20.0):
    lk = w['loglik'] - np.nanmax(w['loglik']); ww = np.exp(lk / scale); ww /= max(ww.sum(), 1e-300)
    return (ww[:, None] * w['cand']).sum(0)

def evaluate(oof_pred):
    errs = []
    for w in WELLS:
        t = w['true']; m = np.isfinite(t)
        if m.any() and w['wid'] in oof_pred:
            errs.append(oof_pred[w['wid']][m] - t[m])
    return pooled_rmse(errs)

RESULTS = {}   # name -> oof_pred(dict wid->traj)

wells: 773 | cand: (96, 3836)


In [2]:
# ===== 特徴量(表形式・候補ごと) + 物理制約 + 局所GR一致 + 系列チャネル + 共通CV =====
import pandas as pd
DATA_DIR = r'C:\Users\kosaka256\Documents\rogii_claude\rogii-wellbore-geology-prediction\train'
NBLK = 6   # eval を NBLK 区間に割って局所GR一致を測る(候補の違いはeval後半に出る→局所が効く)

def load_known(wid):
    try:
        hw = pd.read_csv(f'{DATA_DIR}/{wid}__horizontal_well.csv', usecols=['MD', 'Z', 'TVT_input'])
        kn = hw[hw['TVT_input'].notna()]
        if len(kn) < 3:
            return 0.0, np.nan
        g = (kn['TVT_input'] + kn['Z']).to_numpy(float); md = kn['MD'].to_numpy(float)
        tail = slice(max(0, len(g) - 30), len(g))
        dm = np.diff(md[tail]); dg = np.diff(g[tail]); ok = dm > 0
        kd = float(np.median(dg[ok] / dm[ok])) if ok.sum() >= 3 else 0.0
        return kd, float(g[-1])
    except Exception:
        return 0.0, np.nan
KNOWN = {w['wid']: load_known(w['wid']) for w in WELLS}

BASE_KEYS = ['loglik', 'll_rank', 'resid_mae', 'resid_rms', 'resid_std', 'ncc',
             'd1_std', 'curv', 'slope', 'geo_slope', 'geo_curv',
             'dist_med', 'dist_ens', 'end_minus_start', 'setid', 'eval_len', 'gr_noise',
             'geo_rough', 'dip_var', 'dip_dev', 'anchor_dev', 'clip_frac', 'range_frac', 'wiggle']
BLK_KEYS = [f'b{b}_{s}' for b in range(NBLK) for s in ('rms', 'ncc')]
FEATURE_KEYS = BASE_KEYS + BLK_KEYS
NF = len(FEATURE_KEYS)   # torch.nn.functional の F と名前衝突させない

def well_features(w):
    cand = w['cand']; n, T = cand.shape
    gr = w['gr_raw'].astype(float); z = w['z'].astype(float); md = w['md'].astype(float)
    twt = w['tw_tvt'].astype(float); twg = w['tw_gr'].astype(float)
    twg_at = np.interp(cand.ravel(), twt, twg).reshape(n, T)
    resid = gr[None, :] - twg_at; g0 = (gr - gr.mean()) / (gr.std() + 1e-6)
    ta = (twg_at - twg_at.mean(1, keepdims=True)) / (twg_at.std(1, keepdims=True) + 1e-6)
    d1 = np.diff(cand, axis=1); geo = cand + z[None, :]
    med = np.median(cand, 0); ens = lik_ensemble(w)
    gd = np.gradient(geo, md, axis=1)
    kd, ag = KNOWN[w['wid']]
    twmin, twmax = float(twt.min()), float(twt.max()); twr = max(twmax - twmin, 1.0)
    clip = ((cand <= twmin + 3) | (cand >= twmax - 3)).mean(1)
    wig = (np.abs(np.diff(np.sign(d1), axis=1)) > 0).mean(1)
    base = [w['loglik'].astype(float), rankdata(w['loglik']) / n,
            np.mean(np.abs(resid), 1), np.sqrt(np.mean(resid ** 2, 1)), resid.std(1),
            (ta * g0[None, :]).mean(1), d1.std(1), np.abs(np.diff(cand, 2, axis=1)).mean(1), d1.mean(1),
            np.diff(geo, axis=1).mean(1), np.abs(np.diff(geo, 2, axis=1)).mean(1),
            np.mean(np.abs(cand - med[None, :]), 1), np.mean(np.abs(cand - ens[None, :]), 1),
            cand[:, -1] - cand[:, 0], w['setid'].astype(float), np.full(n, T, float), np.full(n, gr.std(), float),
            np.sqrt(np.mean(np.diff(geo, 2, axis=1) ** 2, 1)), gd.std(1), np.abs(gd.mean(1) - kd),
            np.abs(geo[:, 0] - (ag if np.isfinite(ag) else geo[:, 0])), clip,
            (cand.max(1) - cand.min(1)) / twr, wig]
    # --- 局所GR一致(区間ごと) ---
    edges = np.linspace(0, T, NBLK + 1).astype(int)
    blk = []
    for bi in range(NBLK):
        a0 = edges[bi]; a1 = max(a0 + 1, edges[bi + 1])
        rb = resid[:, a0:a1]
        blk.append(np.sqrt(np.mean(rb ** 2, 1)))                       # 区間残差RMS
        gb = gr[a0:a1]; gb = (gb - gb.mean()) / (gb.std() + 1e-6)
        tb = twg_at[:, a0:a1]; tb = (tb - tb.mean(1, keepdims=True)) / (tb.std(1, keepdims=True) + 1e-6)
        blk.append((tb * gb[None, :]).mean(1))                         # 区間NCC
    X = np.stack(base + blk, 1).astype(np.float32)
    t = w['true']; m = np.isfinite(t)
    y = np.sqrt(np.mean((cand[:, m] - t[None, m]) ** 2, axis=1)) if m.any() else np.full(n, np.nan)
    return X, y.astype(np.float32)

WELL_FEATS = []; X_list, y_list, grp, well_rows = [], [], [], []; row = 0
for wi, w in enumerate(WELLS):
    X, y = well_features(w); WELL_FEATS.append(X)
    X_list.append(X); y_list.append(y); grp.append(np.full(X.shape[0], wi))
    well_rows.append(np.arange(row, row + X.shape[0])); row += X.shape[0]
X_all = np.concatenate(X_list); y_all = np.concatenate(y_list); grp = np.concatenate(grp)
X_all = np.nan_to_num(X_all, nan=0.0, posinf=0.0, neginf=0.0)
MU = X_all.mean(0); SD = X_all.std(0) + 1e-6
print('X:', X_all.shape, '| features:', NF)

uniq = np.arange(len(WELLS))
FOLDS = list(GroupKFold(5).split(uniq, groups=uniq))

CHAN_C = 4   # 系列チャネル数(候補相対, GR残差, 1階差, 2階差=曲率/物理)
def cand_channels(w, sl):
    cand = w['cand'][:, sl].astype(np.float32); n, T = cand.shape
    gr = w['gr_raw'][sl].astype(float); twt = w['tw_tvt'].astype(float); twg = w['tw_gr'].astype(float)
    twg_at = np.interp(cand.ravel(), twt, twg).reshape(n, T)
    g0 = (gr - gr.mean()) / (gr.std() + 1e-6)
    ta = (twg_at - twg_at.mean(1, keepdims=True)) / (twg_at.std(1, keepdims=True) + 1e-6)
    med = np.median(cand, 0)
    d1 = np.diff(cand, axis=1, prepend=cand[:, :1])
    d2 = np.diff(d1, axis=1, prepend=d1[:, :1])
    ch = np.stack([cand - med[None, :], g0[None, :] - ta, d1, d2], 1)
    return np.nan_to_num(ch.astype(np.float32))            # [n,4,T]

def score_to_pred(score, pick='min', k=5, agg='mean'):
    oof = {}
    for wi, w in enumerate(WELLS):
        s = score[well_rows[wi]]; order = np.argsort(s) if pick == 'min' else np.argsort(-s)
        sel = w['cand'][order[:k]]
        oof[w['wid']] = sel.mean(0) if agg == 'mean' else np.median(sel, 0)
    return oof

X: (74208, 36) | features: 36


In [3]:
# ===== oracle & 既存ベースライン =====
def diag():
    eo, ee = [], []
    for w in WELLS:
        t = w['true']; m = np.isfinite(t)
        if not m.any(): continue
        rc = np.sqrt(np.mean((w['cand'][:, m] - t[None, m]) ** 2, 1))
        eo.append(w['cand'][int(np.argmin(rc)), m] - t[m]); ee.append(lik_ensemble(w)[m] - t[m])
    print(f'oracle(単一候補)={pooled_rmse(eo):.3f}  lik-ensemble={pooled_rmse(ee):.3f}')
    return pooled_rmse(eo), pooled_rmse(ee)
ORACLE, LIKENS = diag()
RESULTS['lik-ensemble'] = {w['wid']: lik_ensemble(w) for w in WELLS}

oracle(単一候補)=4.619  lik-ensemble=9.354


In [4]:
# ===== 1-3: LightGBM (回帰 / ランキング / 分類) =====
import lightgbm as lgb
def row_folds():
    for tr, va in FOLDS:
        yield np.isin(grp, tr), np.isin(grp, va)

def m_lgbm_reg():
    oof = np.full(len(y_all), np.nan)
    for trm, vam in row_folds():
        mm = trm & np.isfinite(y_all)
        g = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=63, subsample=0.8, colsample_bytree=0.8)
        g.fit(X_all[mm], y_all[mm]); oof[vam] = g.predict(X_all[vam])
    return score_to_pred(oof, 'min', k=5)

def m_lgbm_rank():
    rel = np.zeros(len(y_all), int)
    for wi in range(len(WELLS)):
        idx = well_rows[wi]; yy = y_all[idx]
        if np.isfinite(yy).any():
            r = rankdata(yy); rel[idx] = np.round((1 - (r - 1) / max(len(r) - 1, 1)) * 4).astype(int)
    oof = np.full(len(y_all), np.nan)
    for trm, vam in row_folds():
        order = np.where(trm)[0]; order = order[np.argsort(grp[order], kind='stable')]
        gs = [int((grp[order] == g).sum()) for g in np.unique(grp[order])]
        g = lgb.LGBMRanker(objective='lambdarank', n_estimators=500, learning_rate=0.03, num_leaves=63)
        g.fit(X_all[order], rel[order], group=gs); oof[vam] = g.predict(X_all[vam])
    return score_to_pred(oof, 'max', k=5)

def m_lgbm_clf():
    lab = np.zeros(len(y_all), int)
    for wi in range(len(WELLS)):
        idx = well_rows[wi]; yy = y_all[idx]
        if np.isfinite(yy).any():
            thr = np.nanpercentile(yy, 20); lab[idx] = (yy <= thr).astype(int)   # 上位20%=good
    oof = np.full(len(y_all), np.nan)
    for trm, vam in row_folds():
        mm = trm & np.isfinite(y_all)
        g = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03, num_leaves=63)
        g.fit(X_all[mm], lab[mm]); oof[vam] = g.predict_proba(X_all[vam])[:, 1]
    return score_to_pred(oof, 'max', k=5)

RESULTS['1.LGBM-reg'] = m_lgbm_reg();  print('1 done', evaluate(RESULTS['1.LGBM-reg']))
RESULTS['2.LGBM-rank'] = m_lgbm_rank(); print('2 done', evaluate(RESULTS['2.LGBM-rank']))
RESULTS['3.LGBM-clf'] = m_lgbm_clf();  print('3 done', evaluate(RESULTS['3.LGBM-clf']))

print(RESULTS)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003057 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8564
[LightGBM] [Info] Number of data points in the train set: 59328, number of used features: 36
[LightGBM] [Info] Start training from score 9.501352


c:\Users\kosaka256\Documents\rogii_claude\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003658 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8555
[LightGBM] [Info] Number of data points in the train set: 59328, number of used features: 36
[LightGBM] [Info] Start training from score 9.445137


c:\Users\kosaka256\Documents\rogii_claude\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003751 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8557
[LightGBM] [Info] Number of data points in the train set: 59328, number of used features: 36
[LightGBM] [Info] Start training from score 9.576658


c:\Users\kosaka256\Documents\rogii_claude\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


... (21 outputs left out)

c:\Users\kosaka256\Documents\rogii_claude\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 12380, number of negative: 47044
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003292 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8566
[LightGBM] [Info] Number of data points in the train set: 59424, number of used features: 36
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.208333 -> initscore=-1.335001
[LightGBM] [Info] Start training from score -1.335001


c:\Users\kosaka256\Documents\rogii_claude\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


3 done 9.840045928955078
{'lik-ensemble': {'000d7d20': array([11747.38559652, 11747.4037735 , 11747.42257333, ...,
       11747.57068942, 11747.55160818, 11747.53387799], shape=(3836,)), '00bbac68': array([12223.5626208 , 12223.58457453, 12223.59408245, ...,
       12216.36196013, 12216.46670148, 12216.58111558], shape=(6014,)), '00e12e8b': array([11604.63547155, 11604.50438458, 11604.41560574, ...,
       11613.82626202, 11613.85566942, 11613.88758717], shape=(4301,)), '015fe0d2': array([12233.83437007, 12233.88103997, 12233.9235717 , ...,
       12262.91915847, 12262.9392802 , 12262.95643531], shape=(4296,)), '01869cd4': array([11745.89031623, 11745.79884457, 11745.73535036, ...,
       11773.15814825, 11773.18999539, 11773.22124947], shape=(5557,)), '01982c1d': array([10694.19574886, 10694.25738858, 10694.30460972, ...,
       10682.68499091, 10682.69646997, 10682.72021052], shape=(3941,)), '028d7b28': array([12427.95363732, 12427.97619035, 12428.00854788, ...,
       12432.08861711

In [5]:
# ===== 共通: 表形式MLP回帰 / 重み学習(blend)トレーナ =====
def t_(a): return torch.tensor(a)

def train_tab_reg(make, epochs=40, bs=4096, lr=2e-3):
    oof = np.full(len(y_all), np.nan)
    Xs = (X_all - MU) / SD
    for trm, vam in row_folds():
        mm = trm & np.isfinite(y_all)
        Xtr = t_(Xs[mm]).to(DEV); ytr = t_(y_all[mm]).to(DEV)
        model = make().to(DEV); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        idx = np.arange(len(ytr))
        for ep in range(epochs):
            np.random.shuffle(idx)
            for c in range(0, len(idx), bs):
                b = idx[c:c + bs]
                pred = model(Xtr[b]).squeeze(-1); loss = F.smooth_l1_loss(pred, ytr[b])
                opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            oof[vam] = model(t_(Xs[vam]).to(DEV)).squeeze(-1).cpu().numpy()
    return score_to_pred(oof, 'min', k=5)

def blend_batch(widx, need_chan=False, crop=None):
    B = len(widx); maxn = max(WELLS[i]['cand'].shape[0] for i in widx)
    Ts = []
    for i in widx:
        T = WELLS[i]['cand'].shape[1]; Ts.append(min(T, crop) if crop else T)
    maxT = max(Ts)
    Xf = np.zeros((B, maxn, NF), np.float32); cand = np.zeros((B, maxn, maxT), np.float32)
    true = np.full((B, maxT), np.nan, np.float32)
    nmask = np.zeros((B, maxn), bool); tmask = np.zeros((B, maxT), bool)
    chan = np.zeros((B, maxn, CHAN_C, maxT), np.float32) if need_chan else None
    for b, i in enumerate(widx):
        w = WELLS[i]; n, T = w['cand'].shape
        if crop and T > crop:
            s = np.random.randint(0, T - crop + 1); sl = slice(s, s + crop); T = crop
        else:
            sl = slice(0, T)
        Xf[b, :n] = (WELL_FEATS[i] - MU) / SD
        cand[b, :n, :T] = w['cand'][:, sl]; true[b, :T] = w['true'][sl]
        nmask[b, :n] = True; tmask[b, :T] = np.isfinite(w['true'][sl])
        if need_chan: chan[b, :n, :, :T] = cand_channels(w, sl)
    out = dict(Xf=t_(Xf).to(DEV), cand=t_(cand).to(DEV), true=t_(true).to(DEV),
               nmask=t_(nmask).to(DEV), tmask=t_(tmask).to(DEV))
    out['chan'] = t_(chan).to(DEV) if need_chan else None
    return out

def train_blend(make, per_md=False, need_chan=False, epochs=15, bs=8, crop=768, lr=2e-3):
    oof = {}
    for tr, va in FOLDS:
        tr = list(tr); model = make().to(DEV); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        for ep in range(epochs):
            np.random.shuffle(tr)
            for c in range(0, len(tr), bs):
                bt = blend_batch(tr[c:c + bs], need_chan, crop)
                logit = model(bt); cand = bt['cand']
                if per_md:
                    logit = logit.masked_fill(~bt['nmask'][:, :, None], -1e9); wgt = torch.softmax(logit, 1)
                    pred = (wgt * cand).sum(1)
                else:
                    logit = logit.masked_fill(~bt['nmask'], -1e9); wgt = torch.softmax(logit, 1)
                    pred = (wgt[:, :, None] * cand).sum(1)
                err = (pred - torch.nan_to_num(bt['true']))[bt['tmask']]
                loss = F.huber_loss(err, torch.zeros_like(err), delta=10.0)
                opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            for i in va:
                bt = blend_batch([i], need_chan, crop=None); logit = model(bt); cand = bt['cand']
                if per_md:
                    logit = logit.masked_fill(~bt['nmask'][:, :, None], -1e9); wgt = torch.softmax(logit, 1)
                    pred = (wgt * cand).sum(1)[0]
                else:
                    logit = logit.masked_fill(~bt['nmask'], -1e9); wgt = torch.softmax(logit, 1)
                    pred = (wgt[:, :, None] * cand).sum(1)[0]
                oof[WELLS[i]['wid']] = pred[:WELLS[i]['cand'].shape[1]].cpu().numpy()
    return oof

In [6]:
# ===== 4-6: MLP-reg / MLP-weight / SetAttn-weight =====
class MLP(nn.Module):
    def __init__(self, fin, h=64):
        super().__init__(); self.net = nn.Sequential(nn.Linear(fin, h), nn.GELU(), nn.Linear(h, h), nn.GELU(), nn.Linear(h, 1))
    def forward(self, x): return self.net(x)

class MLPWeight(nn.Module):
    def __init__(self, fin, h=64):
        super().__init__(); self.net = nn.Sequential(nn.Linear(fin, h), nn.GELU(), nn.Linear(h, h), nn.GELU(), nn.Linear(h, 1))
    def forward(self, bt): return self.net(bt['Xf']).squeeze(-1)            # [B,n]

class SetAttn(nn.Module):
    def __init__(self, fin, d=64, heads=4):
        super().__init__(); self.proj = nn.Linear(fin, d)
        self.attn = nn.MultiheadAttention(d, heads, batch_first=True)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 1))
    def forward(self, bt):
        h = self.proj(bt['Xf']); kp = ~bt['nmask']
        a, _ = self.attn(h, h, h, key_padding_mask=kp); return self.head(h + a).squeeze(-1)

RESULTS['4.MLP-reg'] = train_tab_reg(lambda: MLP(NF)); print('4 done', evaluate(RESULTS['4.MLP-reg']))
RESULTS['5.MLP-weight'] = train_blend(lambda: MLPWeight(NF)); print('5 done', evaluate(RESULTS['5.MLP-weight']))
RESULTS['6.SetAttn-weight'] = train_blend(lambda: SetAttn(NF)); print('6 done', evaluate(RESULTS['6.SetAttn-weight']))

4 done 9.614583015441895
5 done 9.959577560424805
6 done 9.579516410827637


In [8]:
# ===== 7: CNN-cand (候補系列をCNN→RMSE回帰→選択) =====
class CandCNN(nn.Module):
    def __init__(self, cin=CHAN_C, h=32):
        super().__init__()
        self.cnn = nn.Sequential(nn.Conv1d(cin, h, 5, padding=2), nn.GELU(),
                                 nn.Conv1d(h, h, 5, padding=2, dilation=1), nn.GELU(),
                                 nn.Conv1d(h, h, 5, padding=4, dilation=2), nn.GELU())
        self.head = nn.Linear(h, 1)
    def forward_seq(self, x):                 # x:[M,cin,T] -> score[M]
        h = self.cnn(x).mean(-1); return self.head(h).squeeze(-1)

def train_candcnn(epochs=12, bs=6, crop=768, lr=2e-3):
    oof = np.full(len(y_all), np.nan)
    for fi, (tr, va) in enumerate(FOLDS):
        model = CandCNN().to(DEV); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        tr = list(tr)
        for ep in range(epochs):
            np.random.shuffle(tr)
            for c in range(0, len(tr), bs):
                idxs = tr[c:c + bs]; xs, ys = [], []
                for i in idxs:
                    w = WELLS[i]; n, T = w['cand'].shape
                    if not np.isfinite(w['true']).any(): continue
                    sl = slice(0, T) if T <= crop else (lambda s: slice(s, s + crop))(np.random.randint(0, T - crop + 1))
                    ch = cand_channels(w, sl)                       # [n,4,t]
                    tt = w['true'][sl]; mm = np.isfinite(tt)
                    rc = np.sqrt(np.mean((w['cand'][:, sl][:, mm] - tt[mm][None]) ** 2, 1)) if mm.any() else None
                    if rc is None: continue
                    xs.append(t_(ch)); ys.append(t_(rc.astype(np.float32)))
                if not xs: continue
                x = torch.cat(xs, 0).to(DEV); y = torch.cat(ys, 0).to(DEV)
                pred = model.forward_seq(x); loss = F.smooth_l1_loss(pred, y)
                opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            for i in va:
                w = WELLS[i]; T = w['cand'].shape[1]; ch = cand_channels(w, slice(0, T))
                sc = model.forward_seq(t_(ch).to(DEV)).cpu().numpy()
                oof[well_rows[i]] = sc
    return score_to_pred(oof, 'min', k=5)

RESULTS['7.CNN-cand'] = train_candcnn(); print('7 done', evaluate(RESULTS['7.CNN-cand']))

RuntimeError: Sizes of tensors must match except in dimension 0. Expected size 768 but got size 407 for tensor number 5 in the list.

In [9]:
# ===== 8: CNN-permd-gate (各mdで候補に重み→per-md加重=学習版多数決) =====
class GateCNN(nn.Module):
    def __init__(self, cin=CHAN_C, h=32):
        super().__init__()
        self.cnn = nn.Sequential(nn.Conv1d(cin, h, 5, padding=2), nn.GELU(),
                                 nn.Conv1d(h, h, 5, padding=4, dilation=2), nn.GELU(),
                                 nn.Conv1d(h, 1, 1))
    def forward(self, bt):
        chan = bt['chan']; B, n, C, T = chan.shape
        logit = self.cnn(chan.reshape(B * n, C, T)).reshape(B, n, T)   # [B,n,T]
        return logit

RESULTS['8.CNN-permd-gate'] = train_blend(lambda: GateCNN(), per_md=True, need_chan=True, crop=640, bs=6)
print('8 done', evaluate(RESULTS['8.CNN-permd-gate']))

8 done 9.855232238769531


In [10]:
# ===== 9: CNN-traj (候補統計を画像化→CNNでTVT軌跡を直接回帰) =====
def agg_channels(w, sl):
    cand = w['cand'][:, sl].astype(np.float32)
    p = np.percentile(cand, [10, 25, 50, 75, 90], axis=0)   # [5,T]
    ens = lik_ensemble(w)[sl][None, :]
    base = np.median(cand, 0)[None, :]
    ch = np.concatenate([p - base, ens - base], 0)          # 中央値基準の相対 [6,T]
    return ch.astype(np.float32), base[0]

class TrajCNN(nn.Module):
    def __init__(self, cin=6, h=48):
        super().__init__()
        self.cnn = nn.Sequential(nn.Conv1d(cin, h, 5, padding=2), nn.GELU(),
                                 nn.Conv1d(h, h, 5, padding=4, dilation=2), nn.GELU(),
                                 nn.Conv1d(h, h, 5, padding=8, dilation=4), nn.GELU(),
                                 nn.Conv1d(h, 1, 1))
    def forward(self, x): return self.cnn(x).squeeze(1)      # [B,T] = base からの残差

def train_traj(epochs=15, bs=8, crop=768, lr=2e-3):
    oof = {}
    for tr, va in FOLDS:
        tr = list(tr); model = TrajCNN().to(DEV); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        for ep in range(epochs):
            np.random.shuffle(tr)
            for c in range(0, len(tr), bs):
                xs, bs_, ts, ms = [], [], [], []; maxT = 0
                for i in tr[c:c + bs]:
                    w = WELLS[i]; T = w['cand'].shape[1]
                    if not np.isfinite(w['true']).any(): continue
                    sl = slice(0, T) if T <= crop else (lambda s: slice(s, s + crop))(np.random.randint(0, T - crop + 1))
                    ch, base = agg_channels(w, sl); xs.append(ch); bs_.append(base)
                    ts.append(w['true'][sl]); ms.append(np.isfinite(w['true'][sl])); maxT = max(maxT, ch.shape[1])
                if not xs: continue
                B = len(xs); X = np.zeros((B, 6, maxT), np.float32); base = np.zeros((B, maxT), np.float32)
                T_ = np.full((B, maxT), np.nan, np.float32); M = np.zeros((B, maxT), bool)
                for b in range(B):
                    t = xs[b].shape[1]; X[b, :, :t] = xs[b]; base[b, :t] = bs_[b]; T_[b, :t] = ts[b]; M[b, :t] = ms[b]
                X = t_(X).to(DEV); base = t_(base).to(DEV); T_ = t_(T_).to(DEV); M = t_(M).to(DEV)
                pred = base + model(X); err = (pred - torch.nan_to_num(T_))[M]
                loss = F.huber_loss(err, torch.zeros_like(err), delta=10.0)
                opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            for i in va:
                w = WELLS[i]; T = w['cand'].shape[1]; ch, base = agg_channels(w, slice(0, T))
                pred = t_(base).to(DEV) + model(t_(ch[None]).to(DEV))[0]
                oof[w['wid']] = pred.cpu().numpy()
    return oof

RESULTS['9.CNN-traj'] = train_traj(); print('9 done', evaluate(RESULTS['9.CNN-traj']))

9 done 10.804609298706055


In [11]:
# ===== 10: Stacking (1-9のOOFをNNLSブレンド) =====
def stacking():
    names = [k for k in RESULTS if k != 'lik-ensemble']
    rows_true, mats = [], {n: [] for n in names}
    for w in WELLS:
        t = w['true']; m = np.isfinite(t)
        if not m.any() or any(w['wid'] not in RESULTS[n] for n in names): continue
        rows_true.append(t[m])
        for n in names: mats[n].append(RESULTS[n][w['wid']][m])
    yv = np.concatenate(rows_true)
    P = np.stack([np.concatenate(mats[n]) for n in names], 1)   # [rows, M]
    wts, _ = nnls(P, yv); wts = wts / max(wts.sum(), 1e-9)
    print('stacking weights:', {n: round(float(wt), 3) for n, wt in zip(names, wts)})
    oof = {}
    for w in WELLS:
        if any(w['wid'] not in RESULTS[n] for n in names): continue
        oof[w['wid']] = sum(wt * RESULTS[n][w['wid']] for n, wt in zip(names, wts))
    return oof
RESULTS['10.Stacking'] = stacking(); print('10 done', evaluate(RESULTS['10.Stacking']))

stacking weights: {'1.LGBM-reg': 0.136, '2.LGBM-rank': 0.275, '3.LGBM-clf': 0.089, '4.MLP-reg': 0.0, '5.MLP-weight': 0.03, '6.SetAttn-weight': 0.272, '8.CNN-permd-gate': 0.198, '9.CNN-traj': 0.0}
10 done 9.080344223036299


In [12]:
# ===== まとめ =====
print(f'基準: oracle={ORACLE:.3f}  lik-ensemble={LIKENS:.3f}  v9=6.946  RMSE15=無意味')
print('-' * 40)
for name in sorted(RESULTS, key=lambda k: evaluate(RESULTS[k])):
    print(f'  {name:20s} overall RMSE = {evaluate(RESULTS[name]):.3f}')

基準: oracle=4.619  lik-ensemble=9.354  v9=6.946  RMSE15=無意味
----------------------------------------
  10.Stacking          overall RMSE = 9.080
  lik-ensemble         overall RMSE = 9.354
  1.LGBM-reg           overall RMSE = 9.510
  6.SetAttn-weight     overall RMSE = 9.580
  2.LGBM-rank          overall RMSE = 9.605
  4.MLP-reg            overall RMSE = 9.615
  3.LGBM-clf           overall RMSE = 9.840
  8.CNN-permd-gate     overall RMSE = 9.855
  5.MLP-weight         overall RMSE = 9.960
  9.CNN-traj           overall RMSE = 10.805
